This first script sorts the entire sample into training and testing data. By default, it's an 80/20 split.

In [8]:
import pandas as pd
df = pd.read_csv("Filtered_Bugs_Min4_Frac0.55_Gold-x.csv")
print("Unique values in Split_Labels:", df['Split_Labels'].unique())

Unique values in Split_Labels: <StringArray>
['Insect', 'Other']
Length: 2, dtype: str


In [10]:
import pandas as pd
df = pd.read_csv("Filtered_Bugs_Min4_Frac0.55_Gold-x.csv")
print(df['Split_Labels'].value_counts())

Split_Labels
Insect    82
Other      2
Name: count, dtype: int64


## Augmentation & Split

In [24]:
import os
import shutil
import pandas as pd
import random
import torch
from torchvision.transforms import v2
from PIL import Image

BASE_DIR = os.getcwd()
CSV_CANDIDATES = [
    os.path.join(BASE_DIR, "Filtered_Bugs_Min4_Frac0.55_Gold-x.csv"),
]
CSV_FILE = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)

if not CSV_FILE:
    raise FileNotFoundError(f"Could not find a gold CSV file in {BASE_DIR}")

SOURCE_DIR = os.path.join(BASE_DIR, "images")
OUTPUT_DIRS = [
    os.path.join(BASE_DIR, "train_balanced", "Insect"),
    os.path.join(BASE_DIR, "train_balanced", "Other"),
    os.path.join(BASE_DIR, "test", "Insect"),
    os.path.join(BASE_DIR, "test", "Other"),
]

def class_from_label(label):
    return "Insect" if str(label).strip().lower() == "insect" else "Other"

def get_classification(row):
    for col in ("Split_Labels",):
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip():
                return class_from_label(value)
    return "Other"

for folder in OUTPUT_DIRS:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

print(f"Created folders: {', '.join(OUTPUT_DIRS)}")
print(f"Using CSV: {CSV_FILE}")

df = pd.read_csv(CSV_FILE)
file_classes = {}

for _, row in df.iterrows():
    filename = str(row['Filename']).strip()
    if not filename or filename == 'nan':
        continue
    classification = get_classification(row)
    if filename not in file_classes:
        file_classes[filename] = classification
    else:
        if classification == "Insect":
            file_classes[filename] = "Insect"

insects = [f for f, c in file_classes.items() if c == 'Insect']
others = [f for f, c in file_classes.items() if c == 'Other']


NUM_AUGMENTATIONS = 5

augmenter = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    
    v2.RandomChoice([
        v2.RandomRotation([0, 0]), 
        v2.RandomRotation([90, 90]),
        v2.RandomRotation([180, 180]),
        v2.RandomRotation([270, 270])
    ]),
    
    v2.Pad(padding=75, padding_mode='reflect'),
    
    v2.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2)),
    
    # remove padding
    v2.CenterCrop(224),
    
    # MUTUALLY EXCLUSIVE LIGHTING: either blur OR colour jitter
    v2.RandomChoice([
        v2.ColorJitter(brightness=(0.8, 1.2), contrast=0.3, saturation=(0.5, 1.5), hue=0.05),
        v2.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 1.0)),
        v2.Lambda(lambda img: img)
    ]),

    
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.ToPILImage()
])

resizer = v2.Resize((224, 224), antialias=True)

def augment_and_split(files, class_name):
    print(f"\nProcessing {len(files)} raw {class_name} images...")
    all_variations = []
    
    for f in files:
        all_variations.append((f, 'orig'))
        for i in range(NUM_AUGMENTATIONS):
            all_variations.append((f, f'aug_{i}'))
            
  # shuffle the samples to ensure randomness
    random.seed(42)
    random.shuffle(all_variations)
    
    # 80/20 split for training and testing
    train_count = int(len(all_variations) * 0.8)
    train_files = all_variations[:train_count]
    test_files = all_variations[train_count:]
    
    # save images locally
    def process_and_save(item_list, subset_folder):
        count = 0
        for f, v_type in item_list:
            src = os.path.join(SOURCE_DIR, f)
            if not os.path.exists(src):
                continue
                
            img = Image.open(src).convert('RGB')
            
            # apply augmentation or resize if it's the original
            if v_type == 'orig':
                out_img = resizer(img)
            else:
                out_img = augmenter(img)
                
            out_name = f"{f.split('.')[0]}_{v_type}.jpg"
            out_path = os.path.join(BASE_DIR, subset_folder, class_name, out_name)
            out_img.save(out_path)
            count += 1
        return count

    print(f" -> Generating and saving {len(train_files)} images to Train folder...")
    copied_train = process_and_save(train_files, 'train_balanced')
    
    print(f" -> Generating and saving {len(test_files)} images to Test folder...")
    copied_test = process_and_save(test_files, 'test')

    return copied_train, copied_test

print("\n--- STARTING AUGMENTATION & SPLIT ---")
train_ins, test_ins = augment_and_split(insects, 'Insect')
train_oth, test_oth = augment_and_split(others, 'Other')

print("\n--- PROCESS COMPLETE ---")
print(f"Total images/samples: {train_ins + test_ins + train_oth + test_oth}")
print(f"TRAIN SET (80%): {train_ins} Insects | {train_oth} Others")
print(f"TEST SET  (20%): {test_ins} Insects | {test_oth} Others")

Created folders: /Users/ernest/Desktop/RHS Wisley Bug Watch/insect_detection/train_balanced/Insect, /Users/ernest/Desktop/RHS Wisley Bug Watch/insect_detection/train_balanced/Other, /Users/ernest/Desktop/RHS Wisley Bug Watch/insect_detection/test/Insect, /Users/ernest/Desktop/RHS Wisley Bug Watch/insect_detection/test/Other
Using CSV: /Users/ernest/Desktop/RHS Wisley Bug Watch/insect_detection/Filtered_Bugs_Min4_Frac0.55_Gold-x.csv

--- STARTING AUGMENTATION & SPLIT ---

Processing 72 raw Insect images...
 -> Generating and saving 345 images to Train folder...
 -> Generating and saving 87 images to Test folder...

Processing 2 raw Other images...
 -> Generating and saving 9 images to Train folder...
 -> Generating and saving 3 images to Test folder...

--- PROCESS COMPLETE ---
Total images/samples: 402
TRAIN SET (80%): 318 Insects | 9 Others
TEST SET  (20%): 72 Insects | 3 Others


## Training & Evaluation

In [27]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, models
from torchvision.transforms import v2
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. TRANSFORMS (NO AUGMENTATION)
# ==========================================
# Because Cell 1 already created the physical augmentations on your hard drive,
# we ONLY need to convert them into math (Tensors) for the neural network.
standard_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True), # Safety net to ensure correct dimensions
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 2. LOAD DATASETS
# ==========================================
train_root = "train_balanced"
test_root = "test" 

print("Loading datasets...")
train_ds = datasets.ImageFolder(train_root, transform=standard_transform)
test_ds  = datasets.ImageFolder(test_root, transform=standard_transform)

# ==========================================
# 3. TRAINING SETUP
# ==========================================
insect_class_id = train_ds.class_to_idx['Insect']
other_class_id = train_ds.class_to_idx['Other']

targets = np.array(train_ds.targets)
insect_count = np.sum(targets == insect_class_id)
other_count = np.sum(targets == other_class_id)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print("Initializing model...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

weights = torch.tensor([1.0, 1.0], dtype=torch.float)
weights[insect_class_id] = other_count / max(1, insect_count)
weights[other_class_id] = 1.0
weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
EPOCHS = 15 
print(f"Starting training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    train_loss = total_loss / len(train_loader)
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            
            loss = criterion(out, y)
            test_loss += loss.item()
            
            # test accuracy using 50% threshold
            probs = torch.softmax(out, dim=1)
            insect_preds = probs[:, insect_class_id] > 0.5
            final_preds = torch.where(insect_preds, 
                                      torch.tensor(insect_class_id, device=device), 
                                      torch.tensor(other_class_id, device=device))
            correct += (final_preds == y).sum().item()
            total += y.size(0)
            
    test_loss = test_loss / len(test_loader)
    test_acc = correct / total
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}")

# ==========================================
# 7. EVALUATION
# ==========================================
print("\nEvaluating model on test set...")
model.eval()
preds, labels_list = [], []

all_true_labels = []
all_pred_labels = []
all_insect_probs = []
all_other_probs = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        
        probs = torch.softmax(out, dim=1)
        probs_np = probs.cpu().numpy()
        
        insect_preds = (probs[:, insect_class_id] > 0.5).int().cpu().numpy()
        final_preds = np.where(insect_preds == 1, insect_class_id, other_class_id)
        
        preds.extend(final_preds)
        labels_list.extend(y.numpy())
        
        for i in range(len(probs_np)):
            all_insect_probs.append(probs_np[i][insect_class_id])
            all_other_probs.append(probs_np[i][other_class_id])
            
            true_name = 'Insect' if y[i].item() == insect_class_id else 'Other'
            pred_name = 'Insect' if final_preds[i] == insect_class_id else 'Other'
            
            all_true_labels.append(true_name)
            all_pred_labels.append(pred_name)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels_list, preds, target_names=['Insect', 'Other']))

print("\n=== PROBABILITY ARRAY ===")

probability_df = pd.DataFrame({
    'True Label': all_true_labels,
    'Predicted Label': all_pred_labels,
    'Prob: Insect': all_insect_probs,
    'Prob: Other': all_other_probs
})

display(probability_df)

Using device: cpu
Loading datasets...
Initializing model...
Starting training for 15 epochs...
Epoch 1/15 | Train Loss: 0.8605 | Test Loss: 0.7829 | Test Acc: 40.00%
Epoch 2/15 | Train Loss: 0.7510 | Test Loss: 0.6632 | Test Acc: 69.33%
Epoch 3/15 | Train Loss: 0.7512 | Test Loss: 0.6061 | Test Acc: 78.67%
Epoch 4/15 | Train Loss: 0.7138 | Test Loss: 0.6479 | Test Acc: 67.33%
Epoch 5/15 | Train Loss: 0.6557 | Test Loss: 0.6264 | Test Acc: 70.67%
Epoch 6/15 | Train Loss: 0.6555 | Test Loss: 0.6107 | Test Acc: 76.00%
Epoch 7/15 | Train Loss: 0.6350 | Test Loss: 0.5812 | Test Acc: 80.00%
Epoch 8/15 | Train Loss: 0.6514 | Test Loss: 0.5691 | Test Acc: 79.33%
Epoch 9/15 | Train Loss: 0.6007 | Test Loss: 0.5613 | Test Acc: 80.00%
Epoch 10/15 | Train Loss: 0.5771 | Test Loss: 0.5453 | Test Acc: 81.33%
Epoch 11/15 | Train Loss: 0.5793 | Test Loss: 0.5037 | Test Acc: 87.33%
Epoch 12/15 | Train Loss: 0.5597 | Test Loss: 0.5120 | Test Acc: 86.67%
Epoch 13/15 | Train Loss: 0.6096 | Test Loss: 0.48

,True Label,Predicted Label,Prob: Insect,Prob: Other
0,Insect,Insect,0.562005,0.437995
1,Insect,Insect,0.511497,0.488503
2,Insect,Insect,0.759614,0.240386
3,Insect,Insect,0.627613,0.372387
4,Insect,Insect,0.712489,0.287511
...,...,...,...,...
145,Other,Other,0.306532,0.693468
146,Other,Insect,0.523975,0.476025
147,Other,Insect,0.523975,0.476025
148,Other,Other,0.487414,0.512586
